# Day 156 -- Convolutional Neural Networks (CNNs)
## Month 9 | NLP + Deep Learning | Deepanshu Garg

| Field | Details |
|---|---|
| **Day** | 156 (Month 9, Week 1, Day 2) |
| **Topic** | CNNs - Conv2D - MaxPooling - Filters - Feature Maps - Dense vs CNN |
| **Primary Dataset** | MNIST (keras.datasets) -- 70,000 handwritten digit images |
| **Secondary Dataset** | ReviewPulse India (600 rows, seed=155) -- Conv1D demo |
| **Deliverable** | `Day156_CNNs.ipynb` |
| **Max Score** | 80 pts + 10 stars Bonus |
| **Environment** | Google Colab (TF 2.x pre-installed) |

## Month 9 Scorecard

| Day | Topic | Score |
|---|---|---|
| Day 155 | Neural Networks & Keras | **90/90 + 10 stars PERFECT** |
| **Day 156** | **CNNs** | **-- / 80** |


---
## Section 1 -- Raw Data (DO NOT MODIFY)

Loaded below. Never alter these variables.

In [1]:
# RAW DATA LOCK
import numpy as np, pandas as pd, tensorflow as tf, warnings
from tensorflow import keras
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

tf.random.set_seed(156)
np.random.seed(156)
print(f'TensorFlow: {tf.__version__}')

# Dataset A: MNIST
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()
print(f'MNIST train: {X_train_raw.shape}  test: {X_test_raw.shape}')
print(f'Pixel range: {X_train_raw.min()} - {X_train_raw.max()}')

# Dataset B: ReviewPulse India
np.random.seed(155)
n = 600
reviewpulse = pd.DataFrame({
    'project_id'       : range(1, n+1),
    'category'         : np.random.choice(['Web Dev','Data Analysis','Design','Writing','Marketing'], n),
    'platform'         : np.random.choice(['Upwork','Fiverr','Freelancer','Toptal','Guru'], n),
    'budget_inr'       : np.random.randint(5000, 150000, n),
    'duration_days'    : np.random.randint(3, 90, n),
    'num_revisions'    : np.random.randint(0, 10, n),
    'client_rating'    : np.round(np.random.uniform(3.0, 5.0, n), 1),
    'freelancer_exp_yr': np.random.randint(1, 12, n),
    'repeat_client'    : np.random.choice([0,1], n, p=[0.65,0.35]),
    'review_text'      : ['placeholder']*n
})
reviewpulse['high_value'] = (reviewpulse['budget_inr'] > 50000).astype(int)

RAW_MNIST       = (X_train_raw.copy(), y_train_raw.copy(), X_test_raw.copy(), y_test_raw.copy())
RAW_REVIEWPULSE = reviewpulse.copy()
print(f'ReviewPulse: {reviewpulse.shape}')
print(reviewpulse['high_value'].value_counts().to_string())
print('RAW DATA LOCKED')


TensorFlow: 2.20.0
MNIST train: (60000, 28, 28)  test: (10000, 28, 28)
Pixel range: 0 - 255
ReviewPulse: (600, 11)
high_value
1    381
0    219
RAW DATA LOCKED


---
## Section 2 -- Concept Notes

### Why CNNs? The problem with Dense on images

A 28x28 grayscale image = **784 pixels**. A Dense layer treats each pixel as an independent
feature with no concept of neighbourhood. A CNN fixes this with:

**Local connectivity:** each filter sees only a small patch (e.g. 3x3).
**Weight sharing:** the same filter scans every position -- far fewer parameters.
**Hierarchical features:** early layers detect edges; later layers detect digit shapes.

---

### Core building blocks

| Component | What it does |
|---|---|
| `Conv2D(32, (3,3), relu)` | Slides 32 learnable 3x3 filters across the image; each produces one feature map |
| `MaxPooling2D((2,2))` | Takes the max in each 2x2 block; halves spatial dimensions |
| `Flatten()` | Converts 2D feature maps to 1D vector for Dense layers |
| `Dense(10, softmax)` | Final classifier -- same as Day 155 |

---

### How one filter works

```
Image patch (3x3):    Filter (3x3):     -> dot product -> one pixel in feature map
[120, 50, 30]         [ 1,  0, -1]
[ 80, 40, 20]   x     [ 1,  0, -1]
[ 60, 30, 10]         [ 1,  0, -1]
```

32 filters produce 32 feature maps. Early filters detect edges; later filters detect digit curves.

---

### Architecture comparison

```
Dense baseline:
  Input(784) -> Dense(128,relu) -> Dense(64,relu) -> Dense(10,softmax)
  No spatial awareness. Every pixel treated independently.

CNN today:
  Input(28,28,1)
    -> Conv2D(32,(3,3),relu,valid)  -> (26,26,32)
    -> MaxPooling2D(2,2)            -> (13,13,32)
    -> Conv2D(64,(3,3),relu,valid)  -> (11,11,64)
    -> MaxPooling2D(2,2)            -> (5,5,64)
    -> Flatten()                    -> 1600
    -> Dense(128,relu)
    -> Dense(10,softmax)
```

---

### 1D CNN (Task F)

`Conv1D(16, kernel_size=3)` slides over a 1D sequence. Used in NLP and time series.
On **unordered tabular features** it rarely beats a Dense MLP -- this is the key lesson of Task F.

---

### Critical rules
- CNN input must have channel dimension: `.reshape(-1, 28, 28, 1)` not `-1, 28, 28`
- `savefig()` before `plt.show()` or `plt.close()`
- Read printed numbers before writing NRA -- never type from memory


---
## Section 3 -- Practice Tasks

**MNIST training settings for all tasks:** `epochs=10, batch_size=64, validation_split=0.1`


---
### Task A -- Preprocessing (20 pts)

**A1 (10 pts):** Preprocess MNIST for both pipelines:
- Dense: flatten to 784, normalise [0,1] -> `X_train_flat`, `X_test_flat`
- CNN: reshape to (28,28,1), normalise [0,1] -> `X_train_cnn`, `X_test_cnn`
- One-hot encode labels (10 classes) -> `y_train_oh`, `y_test_oh`
- Print all 6 shapes

**A2 (10 pts):** Preprocess ReviewPulse for 1D CNN:
- Features: `budget_inr`, `duration_days`, `num_revisions`, `client_rating`, `freelancer_exp_yr`
- Target: `high_value`
- StandardScaler (fit on train only) then reshape to `(n, 5, 1)`
- Split 80/20, `stratify=y_rp`, `random_state=156`
- Print all 4 shapes


In [2]:
# ---------------------------------------------------------------------
# TASK A -- Preprocessing
# ---------------------------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# A1 -- MNIST
X_train_flat = X_train_raw.reshape(-1, 784).astype('float32') / 255.0
X_test_flat  = X_test_raw.reshape(-1, 784).astype('float32') / 255.0
X_train_cnn  = X_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test_cnn   = X_test_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_train_oh   = to_categorical(y_train_raw, 10)
y_test_oh    = to_categorical(y_test_raw, 10)

print("A1 shapes:")
print(f"X_train_flat: {X_train_flat.shape}  X_test_flat: {X_test_flat.shape}")
print(f"X_train_cnn : {X_train_cnn.shape}   X_test_cnn : {X_test_cnn.shape}")
print(f"y_train_oh  : {y_train_oh.shape}    y_test_oh  : {y_test_oh.shape}")

# A2 -- ReviewPulse
features = ['budget_inr','duration_days','num_revisions','client_rating','freelancer_exp_yr']
X_rp = reviewpulse[features].values
y_rp = reviewpulse['high_value'].values

X_rp_train, X_rp_test, y_rp_train, y_rp_test = train_test_split(
    X_rp, y_rp, test_size=0.2, stratify=y_rp, random_state=156
)

scaler = StandardScaler()
X_rp_train_scaled = scaler.fit_transform(X_rp_train)
X_rp_test_scaled  = scaler.transform(X_rp_test)

X_rp_train_cnn = X_rp_train_scaled.reshape(-1, 5, 1)
X_rp_test_cnn  = X_rp_test_scaled.reshape(-1, 5, 1)

print("\nA2 shapes:")
print(f"X_rp_train_cnn: {X_rp_train_cnn.shape}  X_rp_test_cnn: {X_rp_test_cnn.shape}")
print(f"y_rp_train: {y_rp_train.shape}  y_rp_test: {y_rp_test.shape}")

A1 shapes:
X_train_flat: (60000, 784)  X_test_flat: (10000, 784)
X_train_cnn : (60000, 28, 28, 1)   X_test_cnn : (10000, 28, 28, 1)
y_train_oh  : (60000, 10)    y_test_oh  : (10000, 10)

A2 shapes:
X_rp_train_cnn: (480, 5, 1)  X_rp_test_cnn: (120, 5, 1)
y_rp_train: (480,)  y_rp_test: (120,)


---
### Task B -- Dense Baseline on MNIST (15 pts)

**B1 (8 pts):** Build and train:
```
Input(784) -> Dense(128, relu) -> Dense(64, relu) -> Dense(10, softmax)
```
Compile: `optimizer='adam'`, `loss='categorical_crossentropy'`, `metrics=['accuracy']`
Train 10 epochs. Store as `dense_history`. Print `dense_test_acc` and `dense_model.count_params()`.

**B2 (7 pts):** Plot train AND val accuracy on one figure. Save as `dense_curves.png`.


In [3]:
# ---------------------------------------------------------------------
# TASK B -- Dense Baseline on MNIST
# ---------------------------------------------------------------------
# B1
dense_model = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])
dense_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

dense_history = dense_model.fit(
    X_train_flat, y_train_oh,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

dense_test_acc = dense_model.evaluate(X_test_flat, y_test_oh, verbose=0)[1]
dense_params   = dense_model.count_params()
print(f"Dense test accuracy: {dense_test_acc:.4f}")
print(f"Dense parameters:    {dense_params:,}")

# B2 -- dense_curves.png
plt.figure(figsize=(8,5))
plt.plot(dense_history.history['accuracy'], label='train_acc')
plt.plot(dense_history.history['val_accuracy'], label='val_acc')
plt.title('Dense Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.savefig('dense_curves.png', dpi=150)
plt.close()

Epoch 1/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9160 - loss: 0.2907 - val_accuracy: 0.9587 - val_loss: 0.1438
Epoch 2/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9628 - loss: 0.1255 - val_accuracy: 0.9670 - val_loss: 0.1063
Epoch 3/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9739 - loss: 0.0853 - val_accuracy: 0.9702 - val_loss: 0.0946
Epoch 4/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9813 - loss: 0.0628 - val_accuracy: 0.9738 - val_loss: 0.0881
Epoch 5/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9861 - loss: 0.0476 - val_accuracy: 0.9748 - val_loss: 0.0884
Epoch 6/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.9895 - loss: 0.0365 - val_accuracy: 0.9760 - val_loss: 0.0857
Epoch 7/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9912 - loss: 0.0287 - val_accuracy: 0.9763 - val_loss: 0.0901
Epoch 8/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9926 - loss: 0.0237 - val_accuracy: 0

---
### Task C -- CNN Architecture (20 pts)

**C1 (12 pts):** Build and train exactly this architecture:
```
Input(28,28,1)
-> Conv2D(32, (3,3), activation='relu', padding='valid')
-> MaxPooling2D((2,2))
-> Conv2D(64, (3,3), activation='relu', padding='valid')
-> MaxPooling2D((2,2))
-> Flatten()
-> Dense(128, activation='relu')
-> Dense(10, activation='softmax')
```
Same compile & training settings. Call `cnn_model.summary()`.
Print `cnn_test_acc` and `cnn_model.count_params()`.

**C2 (8 pts):** Plot CNN train AND val accuracy curves. Save as `cnn_curves.png`.


In [4]:
# ---------------------------------------------------------------------
# TASK C -- CNN Architecture
# ---------------------------------------------------------------------
# C1
cnn_model = keras.Sequential([
    keras.layers.Input(shape=(28,28,1)),
    keras.layers.Conv2D(32, (3,3), activation='relu', padding='valid'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64, (3,3), activation='relu', padding='valid'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

cnn_history = cnn_model.fit(
    X_train_cnn, y_train_oh,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

cnn_test_acc = cnn_model.evaluate(X_test_cnn, y_test_oh, verbose=0)[1]
cnn_params   = cnn_model.count_params()
print(f"CNN test accuracy: {cnn_test_acc:.4f}")
print(f"CNN parameters:    {cnn_params:,}")

# C2 -- cnn_curves.png
plt.figure(figsize=(8,5))
plt.plot(cnn_history.history['accuracy'], label='train_acc')
plt.plot(cnn_history.history['val_accuracy'], label='val_acc')
plt.title('CNN Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.savefig('cnn_curves.png', dpi=150)
plt.close()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 48s 55ms/step - accuracy: 0.9511 - loss: 0.1645 - val_accuracy: 0.9813 - val_loss: 0.0669
Epoch 2/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 64s 75ms/step - accuracy: 0.9846 - loss: 0.0501 - val_accuracy: 0.9830 - val_loss: 0.0560
Epoch 3/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 46s 55ms/step - accuracy: 0.9895 - loss: 0.0345 - val_accuracy: 0.9892 - val_loss: 0.0379
Epoch 4/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 83s 55ms/step - accuracy: 0.9929 - loss: 0.0238 - val_accuracy: 0.9897 - val_loss: 0.0350
Epoch 5/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 82s 55ms/step - accuracy: 0.9946 - loss: 0.0174 - val_accuracy: 0.9908 - val_loss: 0.0366
Epoch 6/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 80s 53ms/step - accuracy: 0.9956 - loss: 0.0142 - val_accuracy: 0.9895 - val_loss: 0.0465
Epoch 7/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 45s 53ms/step - accuracy: 0.9965 - loss: 0.0112 - val_accuracy: 0.9900 - val_loss: 0.0407
Epoch 8/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 83s 54ms/step - accuracy: 0.9970 - loss: 0.0088 - 

---
### Task D -- Dense vs CNN Comparison (10 pts)

**D1 (6 pts):** Print a comparison table:

| Model | Test Accuracy | # Parameters |
|---|---|---|
| Dense-only | (from B1) | (count_params) |
| CNN | (from C1) | (count_params) |

**D2 (4 pts):** 2-sentence explanation of WHY CNN beats Dense on MNIST.
Must name at least one specific CNN concept: weight sharing / local receptive field / spatial structure.
Writing only 'CNN is better at images' earns 0 pts.


In [5]:
# ---------------------------------------------------------------------
# TASK D -- Dense vs CNN Comparison
# ---------------------------------------------------------------------
# D1
print("\n--- D1 Comparison Table ---")
print(f"| Model       | Test Accuracy | # Parameters |")
print(f"|-------------|---------------|--------------|")
print(f"| Dense-only  | {dense_test_acc:.4f}       | {dense_params:>10,} |")
print(f"| CNN         | {cnn_test_acc:.4f}       | {cnn_params:>10,} |")

# D2
print("\nD2 -- Explanation:")
print("The CNN outperforms the Dense model because its Conv2D layers employ weight sharing and a local receptive field.")
print("Each filter slides across the image, detecting edges and patterns regardless of position, while using far fewer")
print("parameters than a fully connected layer of the same capacity. This spatial inductive bias is ideal for image data.")


--- D1 Comparison Table ---
| Model       | Test Accuracy | # Parameters |
|-------------|---------------|--------------|
| Dense-only  | 0.9722       |    109,386 |
| CNN         | 0.9910       |    225,034 |

D2 -- Explanation:
The CNN outperforms the Dense model because its Conv2D layers employ weight sharing and a local receptive field.
Each filter slides across the image, detecting edges and patterns regardless of position, while using far fewer
parameters than a fully connected layer of the same capacity. This spatial inductive bias is ideal for image data.


---
### Task E -- Filter & Feature Map Visualisation (10 pts)

**E1 (5 pts):** Extract filters from `cnn_model.layers[0].get_weights()[0]` -> shape (3,3,1,32).
Plot all 32 filters in a **4x8 grid**. Normalise each filter to [0,1] before imshow.
Title: `'Conv2D Layer 1 -- Learned Filters (32 filters, 3x3)'`
Save as `filters_layer1.png`.

**E2 (5 pts):** Test image index 0. Build an intermediate model outputting
`cnn_model.layers[1].output` (= after **first MaxPooling**, shape 13x13x32).
Plot the first 16 feature maps in a **4x4 grid** with `cmap='viridis'`.
Title must include the true digit label. Save as `feature_maps.png`.


In [6]:
# ---------------------------------------------------------------------
# TASK E -- Filter & Feature Map Visualisation
# ---------------------------------------------------------------------
# E1 -- 32 filters from first Conv2D layer
filters = cnn_model.layers[0].get_weights()[0]  # shape (3,3,1,32)
filters_norm = (filters - filters.min()) / (filters.max() - filters.min() + 1e-8)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters_norm[:,:,0,i], cmap='gray')
    ax.axis('off')
plt.suptitle('Conv2D Layer 1 -- Learned Filters (32 filters, 3x3)', fontsize=14)
plt.tight_layout()
plt.savefig('filters_layer1.png', dpi=150)
plt.close()

# E2 -- Feature maps after first MaxPooling (layer index 1)
# Ensure the model is built explicitly
cnn_model.build((None, 28, 28, 1))   # forces input shape

intermediate = keras.Model(
    inputs=cnn_model.inputs[0],      # use .inputs[0] instead of .input
    outputs=cnn_model.layers[1].output
)
sample_img = X_test_cnn[0:1]
true_label = y_test_raw[0]
fmaps = intermediate.predict(sample_img, verbose=0)

fig, axes = plt.subplots(4, 4, figsize=(8,8))
for i, ax in enumerate(axes.flat):
    if i < 16:
        ax.imshow(fmaps[0,:,:,i], cmap='viridis')
    ax.axis('off')
plt.suptitle(f'Feature Maps after First MaxPooling (True digit: {true_label})', fontsize=14)
plt.tight_layout()
plt.savefig('feature_maps.png', dpi=150)
plt.close()

---
### Task F -- 1D CNN on ReviewPulse (5 pts)

Build and train:
```
Input(5,1)
-> Conv1D(16, kernel_size=3, activation='relu', padding='same')
-> Flatten()
-> Dense(32, relu)
-> Dense(1, sigmoid)
```
Compile: `adam`, `binary_crossentropy`, `metrics=['accuracy']`
Train: 20 epochs, batch_size=32, validation_split=0.15
Print `rp_cnn_test_acc`.


In [7]:
# ---------------------------------------------------------------------
# TASK F -- 1D CNN on ReviewPulse
# ---------------------------------------------------------------------
rp_cnn_model = keras.Sequential([
    keras.layers.Input(shape=(5,1)),
    keras.layers.Conv1D(16, kernel_size=3, activation='relu', padding='same'),
    keras.layers.Flatten(),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])
rp_cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

rp_history = rp_cnn_model.fit(
    X_rp_train_cnn, y_rp_train,
    epochs=20,
    batch_size=32,
    validation_split=0.15,
    verbose=1
)

rp_cnn_test_acc = rp_cnn_model.evaluate(X_rp_test_cnn, y_rp_test, verbose=0)[1]
print(f"ReviewPulse 1D CNN test accuracy: {rp_cnn_test_acc:.4f}")

Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.6495 - loss: 0.6393 - val_accuracy: 0.7083 - val_loss: 0.6024
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7108 - loss: 0.5548 - val_accuracy: 0.7361 - val_loss: 0.5322
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7525 - loss: 0.4831 - val_accuracy: 0.7639 - val_loss: 0.4613
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8725 - loss: 0.4117 - val_accuracy: 0.8889 - val_loss: 0.3851
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9534 - loss: 0.3413 - val_accuracy: 0.9444 - val_loss: 0.3119
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9706 - loss: 0.2770 - val_accuracy: 0.9722 - val_loss: 0.2486
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9706 - loss: 0.2234 - val_accuracy: 0.9861 - val_loss: 0.1959
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9657 - loss: 0.1817 - val_accuracy: 0.9861 - val_loss

---
### BONUS Task G -- NRA Business Insights (10 pts)

Write 3 NRA statements. Run all tasks first. Read printed numbers. Then write.

**G1 (3 pts):** CNN vs Dense accuracy gap on MNIST
**G2 (4 pts):** Parameter efficiency -- CNN performance per parameter vs Dense
**G3 (3 pts):** ReviewPulse 1D CNN -- worth it for freelance-scale tabular data?

Each must have `Number:` / `Reason:` / `Action:` headers.
Number must be exact from printed output. Action must name a specific model or threshold.


In [9]:
print("\n--- BONUS NRA STATEMENTS ---")

nra_g1 = f"""NRA G1 -- CNN vs Dense accuracy gap:
  Number: CNN = 0.9910, Dense = 0.9722, Gap = +0.0188 (1.88 percentage points)
  Reason: The CNN's convolutional layers exploit spatial locality – neighbouring pixels are combined via sliding filters, enabling it to learn edge and shape features that the dense model (which treats each pixel independently) cannot capture.
  Action: For production digit recognition, deploy the CNN; for small datasets (< 5k images) or if latency is critical, the dense model may still be acceptable."""

nra_g2 = f"""NRA G2 -- Parameter efficiency:
  Number: CNN uses 225,034 parameters vs. Dense 109,386. Despite having ~2× the parameters, the CNN achieves higher accuracy (0.9910 vs 0.9722). Its effective capacity per parameter is superior because of weight sharing.
  Reason: Weight sharing drastically reduces the number of unique parameters needed to process an image; the same filter is applied everywhere, so the model learns translation‑invariant features without needing separate weights for every patch.
  Action: For image tasks, choose CNN even if parameter count is slightly higher; the gain in accuracy justifies the extra model size. For resource‑constrained environments, we can reduce filters (e.g., 16/32) to shrink the model."""

nra_g3 = f"""NRA G3 -- ReviewPulse 1D CNN:
  Number: Test accuracy = 0.9583 – this is decent on this toy dataset, but the 1D CNN may not be the best choice for unordered tabular features.
  Reason: The 1D CNN assumes local sequential structure across the feature vector (kernel_size=3), but our tabular features (budget, duration, revisions, etc.) are unordered. The Conv1D forces a meaningless neighbourhood assumption, which may degrade performance compared to a plain MLP or tree‑based model on larger, noisier data.
  Action: Do not use Conv1D for unordered tabular data unless the features have a natural order (e.g., time series). Instead, use a Dense MLP with batch normalization or a tree‑based model (XGBoost) for freelance‑scale prediction tasks. Retain this 1D CNN only for demonstration purposes."""

print(nra_g1)
print("\n" + nra_g2)
print("\n" + nra_g3)


--- BONUS NRA STATEMENTS ---
NRA G1 -- CNN vs Dense accuracy gap:
  Number: CNN = 0.9910, Dense = 0.9722, Gap = +0.0188 (1.88 percentage points)
  Reason: The CNN's convolutional layers exploit spatial locality – neighbouring pixels are combined via sliding filters, enabling it to learn edge and shape features that the dense model (which treats each pixel independently) cannot capture.
  Action: For production digit recognition, deploy the CNN; for small datasets (< 5k images) or if latency is critical, the dense model may still be acceptable.

NRA G2 -- Parameter efficiency:
  Number: CNN uses 225,034 parameters vs. Dense 109,386. Despite having ~2× the parameters, the CNN achieves higher accuracy (0.9910 vs 0.9722). Its effective capacity per parameter is superior because of weight sharing.
  Reason: Weight sharing drastically reduces the number of unique parameters needed to process an image; the same filter is applied everywhere, so the model learns translation‑invariant features 

---
## Section 4 -- Scoring Rubric

| Task | Subtask | Pts | What is checked |
|---|---|---|---|
| **A** | A1 MNIST preprocessing | 10 | 6 correct shapes; float32; /255; one-hot 10 classes |
| **A** | A2 ReviewPulse preprocessing | 10 | 5 features; scaler fit on train only; reshaped (n,5,1); stratify |
| **B** | B1 Dense model + train | 8 | Architecture matches spec; 10 epochs; dense_test_acc + count_params printed |
| **B** | B2 Dense curves | 7 | Train AND val plotted; labelled; dense_curves.png saved |
| **C** | C1 CNN + train | 12 | Exact architecture; summary() called; cnn_test_acc + count_params printed |
| **C** | C2 CNN curves | 8 | Both curves labelled; cnn_curves.png saved |
| **D** | D1 Comparison table | 6 | Values from printed output; count_params() used |
| **D** | D2 Written explanation | 4 | Must name weight sharing OR local receptive field -- not just 'CNN is better' |
| **E** | E1 Filter visualisation | 5 | layers[0]; 4x8 grid; normalised; filters_layer1.png saved |
| **E** | E2 Feature map visualisation | 5 | layers[1] = MaxPooling (NOT Conv2D); 4x4 grid; title has digit; saved |
| **F** | 1D CNN ReviewPulse | 5 | Conv1D spec; 20 epochs; rp_cnn_test_acc printed |
| **G star** | G1 NRA | 3 | Exact printed numbers; Reason names Conv2D mechanism |
| **G star** | G2 NRA | 4 | count_params() numbers in Number; weight sharing in Reason |
| **G star** | G3 NRA | 3 | Printed accuracy in Number; Action recommends specific alternative |
| **TOTAL** | | **80 + 10 stars** | |

### Common mistakes
- E2: `layers[0]` (Conv2D output) instead of `layers[1]` (MaxPooling output) -> -2 pts
- D2: 'CNN has spatial awareness' without explaining HOW -> -2 pts
- A1: `.reshape(-1,28,28)` missing channel dim -> CNN pipeline will fail
- G NRA: Number not matching printed output -> -1 per NRA

---

## Interview Answer -- Day 156

**'What is a CNN and when would you use it?'**

> A CNN uses Conv2D layers that slide learnable filters across an image to detect local
> patterns such as edges and curves through weight sharing. One filter of size 3x3 is
> reused at every pixel position rather than allocating unique weights per location,
> making CNNs more parameter-efficient and spatially aware than Dense networks on image data.
> I would use a CNN for image classification tasks such as product defect detection or
> document image routing. For tabular client data I default to XGBoost or a Dense MLP --
> the spatial locality assumption does not apply and simpler models are easier to explain.
